In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from crewai import Agent
from crewai_tools import SerperDevTool

from langchain_openai import ChatOpenAI

search_tool = SerperDevTool()

# Define your agents with roles and goals
# researcher = Agent(
# 	role='Senior Research Analyst',
# 	goal='Uncover cutting-edge developments in AI and data science',
# 	backstory="""You work at a leading tech think tank.
# 	Your expertise lies in identifying emerging trends.
# 	You have a knack for dissecting complex data and presenting actionable insights.""",
# 	verbose=True,
# 	allow_delegation=False,
# 	llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7),
# 	tools=[search_tool]
# )
researcher = Agent(
	role='수석 연구 분석가',
	goal='AI 및 데이터 과학의 최첨단 발전 사항 발견',
	backstory="""당신은 선도적인 기술 싱크탱크에서 일하고 있습니다.
	당신의 전문 분야는 신흥 트렌드를 식별하는 것입니다.
	복잡한 데이터를 분석하고 실행 가능한 인사이트를 제공하는 데 뛰어납니다.""",
	verbose=True,
	allow_delegation=False,
	llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7),
	tools=[search_tool]
)
# writer = Agent(
# 	role='Tech Content Strategist',
# 	goal='Craft compelling content on tech advancements',
# 	backstory="""You are a renowned Content Strategist, known for your insightful and engaging articles.
# 	You transform complex concepts into compelling narratives.""",
# 	verbose=True,
# 	llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7),
# 	allow_delegation=True
# )
writer = Agent(
	role='기술 콘텐츠 전략가',
	goal='기술 발전에 대한 설득력 있는 콘텐츠 작성',
	backstory="""당신은 통찰력 있고 매력적인 기사로 유명한 콘텐츠 전략가입니다.
	복잡한 개념을 설득력 있는 이야기로 변환하는 능력이 있습니다.""",
	verbose=True,
	llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7),
	allow_delegation=True
)

translator = Agent(
	role='번역 전문가',
	goal='최종 결과물을 한국어로 정확하게 번역',
	backstory="""당신은 다년간의 번역 경험을 가진 전문가입니다.
	복잡한 기술 문서도 정확하고 매끄럽게 번역할 수 있습니다.""",
	verbose=True,
	llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0.7),
	allow_delegation=False
)

In [3]:
from crewai import Task


research_task = Task(
    description="""2024년 AI의 최신 발전 사항에 대한 포괄적인 분석을 수행하세요.
    주요 트렌드, 획기적인 기술, 잠재적인 산업 영향을 식별하세요.""",
    expected_output="주요 사항을 요약한 전체 분석 보고서",
    agent=researcher
)

blog_post_task = Task(
    description="""제공된 인사이트를 활용하여 가장 중요한 AI 발전 사항을 강조하는
    흥미로운 블로그 포스트를 작성하세요.
    포스트는 기술에 정통한 독자를 위해 정보 전달력이 있으면서도 접근 가능해야 합니다.
    멋지게 들리도록 작성하되, 너무 복잡한 단어는 피해서 AI처럼 들리지 않도록 하세요.""",
    expected_output="최소 4개 단락의 완성된 블로그 포스트",
    agent=writer,
    context=[
        research_task
    ]
)

translation_task = Task(
    description="""작성된 블로그 포스트를 한국어로 정확하게 번역하세요.
    기술적인 내용도 매끄럽게 번역되도록 주의하세요.""",
    expected_output="한국어로 번역된 블로그 포스트",
    agent=translator,
    context=[
        blog_post_task
    ],
    output_file="2024_AI_Tech_Trends.md"
)


In [4]:
from crewai import Crew, Process


# Instantiate your crew with a sequential process
crew = Crew(
    agents=[researcher, writer, translator],
    tasks=[
        research_task,
        blog_post_task, 
        translation_task],
    verbose=2, # You can set it to 1 or 2 to different logging levels
    process = Process.sequential
)

# Get your crew to work!
result = crew.kickoff()

print("######################")
print(result)

 [2024-08-05 20:57:38][DEBUG]: == Working Agent: 수석 연구 분석가
 [2024-08-05 20:57:38][INFO]: == Starting Task: 2024년 AI의 최신 발전 사항에 대한 포괄적인 분석을 수행하세요.
    주요 트렌드, 획기적인 기술, 잠재적인 산업 영향을 식별하세요.


> Entering new CrewAgentExecutor chain...
Action: Search the internet  
Action Input: {"search_query": "2024 AI latest advancements trends groundbreaking technologies industry impact"}   


Search results: Title: McKinsey technology trends outlook 2024
Link: https://www.mckinsey.com/capabilities/mckinsey-digital/our-insights/the-top-trends-in-tech
Snippet: Which new technology will have the most impact in 2024 and beyond ... Advancements in AI are ushering in a new era of more capable robots ...
---
Title: 24 New Technology Trends in 2024: Exploring the Future
Link: https://www.simplilearn.com/top-technology-trends-and-jobs-article
Snippet: 24 New Technology Trends in 2024: Exploring the Future · 1. AI-Generated Content · 2. Quantum Computing · 3. 5G Expansion · 4. Virtual Reality (VR) ...
---
Title: 

In [9]:
print(result)

**제목: 2024년의 혁신적인 AI 발전: 주요 트렌드와 기술**

2024년이 진행됨에 따라 인공지능(AI)의 환경은 전례 없는 속도로 진화하고 있습니다. 올해는 AI가 단순한 도구가 아니라 산업을 재편하고 혁신을 주도하며 인력 동태에 영향을 미치는 변혁의 힘으로 자리잡는 중요한 전환점을 의미합니다. 아래에서는 2024년의 가장 중요한 AI 발전을 살펴보며 주요 트렌드, 혁신적인 기술 및 다양한 분야에 미치는 깊은 영향을 강조합니다.

### 1. 생성적 AI: 민주화 및 실용적 응용

생성적 AI는 2024년 기술 발전의 최전선에 남아 있습니다. 이 기술의 응용은 단순한 신선함을 넘어 실용적이고 일상적인 사용으로 확장되고 있으며, 기업에 창의성과 생산성을 향상시키는 도구를 제공합니다. 기업들은 콘텐츠 제작, 디자인 및 소프트웨어 개발을 위해 생성 모델을 활용하고 있습니다. 이 기술이 더 접근 가능해짐에 따라 AI 도구의 민주화는 비기술적 사용자들 사이에 새로운 혁신의 물결을 일으키고 있습니다. 현재의 도전 과제는 이러한 모델이 사용자에게 실제 가치를 제공하는 동시에 윤리적 기준을 유지하는 것입니다.

### 2. 다중 모달 AI: 데이터 유형의 통합

또 다른 중요한 트렌드는 다양한 유형의 데이터—텍스트, 이미지, 오디오 및 비디오—를 단일 프레임워크로 통합하는 다중 모달 AI 시스템의 부상입니다. 이 기능은 더 풍부한 상호 작용과 더 미묘한 이해를 가능하게 하여 의료 진단, 고객 서비스 및 창의 산업 등 다양한 분야에서의 응용을 향상시킵니다. 서로 다른 데이터 모달리티를 결합함으로써, 조직들은 새로운 통찰력을 열어주고 의사 결정 과정을 개선할 수 있습니다.

### 3. 소형 언어 모델 및 오픈 소스 이니셔티브

2024년에는 더 작고 효율적인 언어 모델로의 전환이 두드러집니다. 이러한 모델은 제한된 컴퓨팅 자원으로 효과적으로 작동하도록 설계되어, AI를 소규모 조직과 스타트업에도 더 접근 가능하게 만듭니다. 오픈 소스 AI 이니셔티브도 주목받고 있으며